# QUERY 5
Cantidad de transacciones del período [2022-09-01, 2022-09-05] con formato de pago
"Wire" o "ACH" cuyo monto convertido a USD sea menor a 1

In [1]:
import numpy as np
import pandas as pd
import requests

RUTA_DE_DATASETS = "../../datasets"
NOMBRE_ARCHIVO = "transacciones_sample.csv"
SOLUCION = "q5_solucion.csv"

transacciones = pd.read_csv(f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO}")
print(f"Total filas cargadas: {len(transacciones)}")
transacciones.head()


Total filas cargadas: 1030


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/02 01:08,3653,80BB91590,24216,810CF78E0,791777.14,US Dollar,791777.14,US Dollar,Cheque,1
1,2022/09/02 02:15,3653,80BB91590,24217,810CF78E1,520000.00,US Dollar,520000.00,US Dollar,Wire,1
2,2022/09/03 03:30,3653,80BB91590,24218,810CF78E2,480000.00,US Dollar,480000.00,US Dollar,ACH,1
3,2022/09/04 04:45,3653,80BB91590,24219,810CF78E3,610000.00,US Dollar,610000.00,US Dollar,Cheque,1
4,2022/09/05 05:50,3653,80BB91590,24220,810CF78E4,455000.00,US Dollar,455000.00,US Dollar,Cash,1


In [2]:
# Filtro por período [2022-09-01, 2022-09-05]
# "2022/09/01" <= Timestamp <= "2022/09/05" (inclusive)
INICIO = "2022/09/01"
FIN    = "2022/09/06"  # "2022/09/06" para incluir todo el día 5/9

filtro_periodo = transacciones[
    (transacciones["Timestamp"] >= INICIO) &
    (transacciones["Timestamp"] < FIN)
]
print(f"Transacciones en el período: {len(filtro_periodo)}")
filtro_periodo.head()

Transacciones en el período: 562


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/02 01:08,3653,80BB91590,24216,810CF78E0,791777.14,US Dollar,791777.14,US Dollar,Cheque,1
1,2022/09/02 02:15,3653,80BB91590,24217,810CF78E1,520000.00,US Dollar,520000.00,US Dollar,Wire,1
2,2022/09/03 03:30,3653,80BB91590,24218,810CF78E2,480000.00,US Dollar,480000.00,US Dollar,ACH,1
3,2022/09/04 04:45,3653,80BB91590,24219,810CF78E3,610000.00,US Dollar,610000.00,US Dollar,Cheque,1
4,2022/09/05 05:50,3653,80BB91590,24220,810CF78E4,455000.00,US Dollar,455000.00,US Dollar,Cash,1


In [3]:
# Filtro por formato de pago: Wire o ACH
FORMATOS = ["Wire", "ACH"]

filtro_formato = filtro_periodo[filtro_periodo["Payment Format"].isin(FORMATOS)]
print(f"Transacciones Wire/ACH en el período: {len(filtro_formato)}")
filtro_formato.head()


Transacciones Wire/ACH en el período: 68


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
1,2022/09/02 02:15,3653,80BB91590,24217,810CF78E1,520000.00,US Dollar,520000.00,US Dollar,Wire,1
2,2022/09/03 03:30,3653,80BB91590,24218,810CF78E2,480000.00,US Dollar,480000.00,US Dollar,ACH,1
6,2022/09/02 09:25,24217,810CF78E1,99999,81FINAL990,519000.00,US Dollar,519000.00,US Dollar,Wire,1
7,2022/09/03 10:40,24218,810CF78E2,99999,81FINAL990,479000.00,US Dollar,479000.00,US Dollar,ACH,1
43,2022/09/01 15:37,5923,8028C7C00,17213,802BDF150,1947.41,Yuan,1947.41,Yuan,ACH,0


In [4]:
# Descarga de cotizaciones desde API Frankfurter (base EUR)
# y forward-fill para días sin mercado (fines de semana / feriados)
from datetime import date, timedelta

url = "https://api.frankfurter.app/2022-09-01..2022-09-05"
resp = requests.get(url, timeout=10)
resp.raise_for_status()
cotizaciones_raw = resp.json()["rates"]   # solo días hábiles
print("Fechas con cotización de la API:", list(cotizaciones_raw.keys()))

# Expandir el rango completo usando la última cotización conocida
cotizaciones = {}
last_rates = None
dia = date(2022, 9, 1)
fin = date(2022, 9, 5)
while dia <= fin:
    key = dia.strftime("%Y-%m-%d")
    if key in cotizaciones_raw:
        last_rates = cotizaciones_raw[key]
    if last_rates is not None:
        cotizaciones[key] = last_rates
    dia += timedelta(days=1)

print("Fechas disponibles tras forward-fill:", list(cotizaciones.keys()))


Fechas con cotización de la API: ['2022-09-01', '2022-09-02', '2022-09-05']
Fechas disponibles tras forward-fill: ['2022-09-01', '2022-09-02', '2022-09-03', '2022-09-04', '2022-09-05']


In [5]:
# Conversión de monto recibido a USD (misma lógica que el worker distribuido)
CURRENCY_MAP = {
    "US Dollar":         "USD",
    "Euro":              "EUR",
    "UK Pound":          "GBP",
    "Yen":               "JPY",
    "Australian Dollar": "AUD",
    "Bitcoin":           "BTC",
    "Brazil Real":       "BRL",
    "Canadian Dollar":   "CAD",
    "Mexican Peso":      "MXN",
    "Ruble":             "RUB",
    "Rupee":             "INR",
    "Saudi Riyal":       "SAR",
    "Shekel":            "ILS",
    "Swiss Franc":       "CHF",
    "Yuan":              "CNY",
}

def convertir_a_usd(row):
    iso = CURRENCY_MAP.get(row["Receiving Currency"])
    if not iso:
        return None
    if iso == "USD":
        return float(row["Amount Received"])

    fecha = row["Timestamp"].split(" ")[0].replace("/", "-")
    rates = cotizaciones.get(fecha)  # ya tiene forward-fill para fines de semana
    if not rates:
        return None

    rate_usd = rates.get("USD")
    if not rate_usd:
        return None

    monto = float(row["Amount Received"])
    if iso == "EUR":
        return monto * rate_usd

    # Triangulación: origen → EUR → USD
    rate_origen = rates.get(iso)
    if not rate_origen:
        return None
    return (monto / rate_origen) * rate_usd

filtro_formato = filtro_formato.copy()
filtro_formato["amount_usd"] = filtro_formato.apply(convertir_a_usd, axis=1)

# Descartamos filas sin cotización (divisa no mapeada) y filtramos monto < 1 USD
menores_1_usd = filtro_formato.dropna(subset=["amount_usd"])
menores_1_usd = menores_1_usd[menores_1_usd["amount_usd"] < 1.0]
print(f"Transacciones < 1 USD: {len(menores_1_usd)}")
menores_1_usd[["Timestamp", "Receiving Currency", "Amount Received", "amount_usd"]].head()


Transacciones < 1 USD: 3


,Timestamp,Receiving Currency,Amount Received,amount_usd
683,2022/09/03 06:50,Euro,0.20,0.19986
758,2022/09/05 07:54,Euro,0.04,0.03968
763,2022/09/02 19:19,US Dollar,0.96,0.96000


In [6]:
# Resultado final: cantidad de transacciones
count = len(menores_1_usd)
resultado = pd.DataFrame({"count": [count]})
print(f"Q5 resultado: {count} transacciones")

resultado.to_csv(SOLUCION, index=False)
print(f"Guardado en {SOLUCION}")


Q5 resultado: 3 transacciones
Guardado en q5_solucion.csv
